# Create BioASQ retrieval subsets

This notebook combines the expert-authored BioASQ 11b questions and gold PubMed document annotations with a corpus of PubMed titles and abstracts. It can create separate retrieval samples by question type and by whether a question has one or multiple relevant documents.

Only questions whose complete gold-document set is present in the corpus are eligible. This avoids silently converting a multi-document question into a single-document question.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets tqdm


In [ ]:
import logging

from src.dataset_builder import (
    create_bioasq_calibration_set,
    create_bioasq_sample,
)
from src.io import load_bioasq_benchmark, mount_google_drive

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("bioasq-sample")


## Configuration

`N_QUERIES_PER_SUBSET = 90` keeps all six subsets equally sized. Set it to `None` to use every eligible question in each subset.

In [ ]:
CORPUS_DATASET_NAME = "DinoStackAI/bioasq-rag-13b-resplit"
CORPUS_CONFIG = "corpus"
CORPUS_SPLIT = "train"
QUESTIONS_SOURCE = "https://zenodo.org/api/records/7655130/files/training11b.json/content"

N_QUERIES_PER_SUBSET = 90
N_CORPUS_DOCS = 30_000
N_CALIBRATION_DOCS = 5_000
SEED = 42
CALIBRATION_SEED = 43

OUTPUT_ROOT = "/content/drive/MyDrive/Retreaval/data"
CALIBRATION_OUTPUT_DIR = (
    "/content/drive/MyDrive/Retreaval/calibration/bioasq-5k"
)
CALIBRATION_QUESTION_TYPES = ("list", "factoid", "summary")


## Load the shared benchmark once

The source corpus is loaded only once and reused by the calibration and subset-creation blocks. Empty source documents are discarded while loading.

In [ ]:
mount_google_drive()
benchmark = load_bioasq_benchmark(
    corpus_dataset_name=CORPUS_DATASET_NAME,
    corpus_config=CORPUS_CONFIG,
    corpus_split=CORPUS_SPLIT,
    questions_source=QUESTIONS_SOURCE,
)


## Create the shared calibration set

The calibration set contains 5,000 documents that are not relevant to any complete `list`, `factoid`, or `summary` question. It is stored outside the retrieval-dataset directory and excluded from every sampled evaluation corpus.

In [ ]:
created_calibration = create_bioasq_calibration_set(
    benchmark,
    n_documents=N_CALIBRATION_DOCS,
    seed=CALIBRATION_SEED,
    output_dir=CALIBRATION_OUTPUT_DIR,
    protected_question_types=CALIBRATION_QUESTION_TYPES,
)
calibration_document_ids = frozenset(
    created_calibration.calibration_set.corpus
)
print(
    f"Created calibration set: {created_calibration.output_dir} "
    f"({len(calibration_document_ids)} documents)"
)


## Shared builder arguments

The creation functions are imported from `src.dataset_builder`. This cell collects the arguments shared by all six retrieval subsets.

In [ ]:
sample_creation_kwargs = {
    "benchmark": benchmark,
    "n_queries": N_QUERIES_PER_SUBSET,
    "n_corpus_docs": N_CORPUS_DOCS,
    "seed": SEED,
    "output_root": OUTPUT_ROOT,
    "calibration_document_ids": calibration_document_ids,
    "calibration_set_path": created_calibration.output_dir,
}


## Create `list` subsets

This block creates `list-one` and `list-multiple`.

In [ ]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"list-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="list",
        documents=document_filter,
        subset_name=key,
        **sample_creation_kwargs,
    )
    print(
        f"Created {key}: {created_subsets[key].output_dir} "
        f"({created_subsets[key].dataset_id})"
    )


## Create `factoid` subsets

This block creates `factoid-one` and `factoid-multiple`.

In [ ]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"factoid-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="factoid",
        documents=document_filter,
        subset_name=key,
        **sample_creation_kwargs,
    )
    print(
        f"Created {key}: {created_subsets[key].output_dir} "
        f"({created_subsets[key].dataset_id})"
    )


## Create `summary` subsets

This block creates `summary-one` and `summary-multiple`.

In [ ]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"summary-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="summary",
        documents=document_filter,
        subset_name=key,
        **sample_creation_kwargs,
    )
    print(
        f"Created {key}: {created_subsets[key].output_dir} "
        f"({created_subsets[key].dataset_id})"
    )


## Browse subset examples

Change `EXAMPLE_SUBSET` and `EXAMPLE_PAGE`, then rerun the cell. The selected subset must have been created in this runtime.

In [ ]:
from IPython.display import Markdown, display

EXAMPLE_SUBSET = "list-multiple"
EXAMPLE_PAGE = 0
EXAMPLES_PER_PAGE = 3

if EXAMPLE_SUBSET not in created_subsets:
    raise KeyError(f"Create {EXAMPLE_SUBSET!r} before browsing it")
created_sample = created_subsets[EXAMPLE_SUBSET]
sample = created_sample.sample
queries = sample.queries
relevant_docs = sample.relevant_docs
corpus = sample.corpus
metadata = sample.metadata
query_items = list(queries.items())
n_pages = max(1, (len(query_items) + EXAMPLES_PER_PAGE - 1) // EXAMPLES_PER_PAGE)
page = EXAMPLE_PAGE % n_pages
start = page * EXAMPLES_PER_PAGE

display(Markdown(
    f"### {EXAMPLE_SUBSET}: page {page + 1} of {n_pages}  "
    f"\nEligible questions: {metadata['n_eligible_queries']}"
))
for qid, query in query_items[start:start + EXAMPLES_PER_PAGE]:
    snippets = []
    for doc_id in sorted(relevant_docs[qid]):
        text = corpus[doc_id].replace("\n", " ")
        suffix = "…" if len(text) > 700 else ""
        snippets.append(f"**Document `{doc_id}`:** {text[:700]}{suffix}")
    display(Markdown(
        f"**Query `{qid}` ({metadata['query_types'][qid]}):** {query}\n\n"
        + "\n\n".join(snippets)
    ))
